# Cross-Architecture Test — Does amnesic work in Qwen3.5-4B dense?

**Question**: is the +2.7pp L17 amnesic effect specific to hybrid MoE+GDN+Gated-Attn, or does it generalize to dense transformers?

**Test**: run the same amnesic protocol on **Qwen3.5-4B** (dense, 36 layers, standard transformer).

**Protocol** (standalone, ~15-20 min):
1. Load Qwen3.5-4B (~8 GB bf16)
2. Feed our 200 SuperGPQA 10-option questions, capture L15 residual at last prompt token
3. Split 70/30 train/test
4. Fit probe on train (gold letter as label), build 10 letter-direction vectors
5. Eval on test: baseline vs amnesic_all (project out 10 letter directions at last prompt token)
6. Also sweep layers L10, L15, L20 for best amnesic position

**Outcomes**:
- **Dense shows +X pp with X ≥ 2.7**: amnesic generalizes across architectures → paper has broad claim
- **Dense shows 0pp or negative**: amnesic is hybrid-specific → paper ties the +2.7pp to hybrid arch uniquely
- **Dense shows large positive (+5pp+)**: amnesic is actually stronger in dense, motivates further dense study

Any outcome is publication-relevant.

## Cell 1 — Install + setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
except Exception:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer',
        'transformers', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/dense_amnesic'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.5-4B dense

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
from safetensors import safe_open

# Use Qwen2.5-7B-Instruct as a stand-in if Qwen3.5-4B not available
# Adjust MODEL_ID to whatever dense Qwen is accessible
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # dense, 28 layers
# Fallback candidates: Qwen/Qwen2.5-3B-Instruct (36 layers), Qwen/Qwen2.5-1.5B-Instruct

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','layers'), ('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Count total layers
def get_all_layers(m):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','layers'), ('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('layers not found')

all_layers = get_all_layers(model)
N_LAYERS = len(all_layers)
print(f'Model: {MODEL_ID}')
print(f'Layers: {N_LAYERS}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Pick 3 layers spread through depth for probe training
PROBE_LAYERS = [N_LAYERS // 3, N_LAYERS // 2, (2 * N_LAYERS) // 3]
print(f'Probe layers (1/3, 1/2, 2/3 depth): {PROBE_LAYERS}')

## Cell 3 — Load our SuperGPQA 10-opt questions (from Stage B metadata)

In [ ]:
# Reuse Stage B dataset for questions + options (not activations — those are Qwen3.6-specific)
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        opts = json.loads(meta['options'])
        if len(opts) == 10:
            questions.append(dict(
                question=meta['question'],
                options=opts,
                gold_letter=meta['gold_letter'],
                n_options=10))
print(f'{len(questions)} 10-option questions loaded')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}
print('letter_ids:', letter_ids)

## Cell 4 — Collect activations + baseline predictions

For each question, capture residual at each of `PROBE_LAYERS` at the last prompt token. Also record baseline prediction (argmax of letter logits).

In [ ]:
# Hooks to capture residual at probe layers
captured = {L: None for L in PROBE_LAYERS}
def make_capture(L):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        # Last token of this forward pass (prefill: T == prompt_len, so last is last prompt token)
        captured[L] = h[0, -1, :].detach().clone()
    return hook

cap_handles = []
for L in PROBE_LAYERS:
    cap_handles.append(get_layer_module(model, L).register_forward_hook(make_capture(L)))

random.seed(42)
random.shuffle(questions)

rows = []
t0 = time.time()
for i, q in enumerate(tqdm(questions, desc='collect')):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        logits = out.logits[0, -1]
        letter_logits = {L: logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:q['n_options']]}
        pred = max(letter_logits, key=letter_logits.get)
        row = dict(
            idx=i,
            gold=q['gold_letter'],
            baseline_pred=pred,
            baseline_correct=(pred == q['gold_letter']),
            question=q['question'],
            options=q['options'],
            n_options=q['n_options'],
            prompt_len=ids.shape[1],
        )
        for L in PROBE_LAYERS:
            row[f'act_L{L}'] = captured[L].float().cpu().numpy()
        rows.append(row)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

for h in cap_handles: h.remove()

baseline_acc = sum(r['baseline_correct'] for r in rows) / len(rows) * 100
print(f'\nBaseline accuracy: {baseline_acc:.1f}% (N={len(rows)})')
print(f'Time: {(time.time()-t0)/60:.1f} min')

## Cell 5 — Fit probe per layer + build letter directions

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Split 70/30
idxs = list(range(len(rows)))
random.Random(42).shuffle(idxs)
cut = int(0.7 * len(idxs))
train_idx, test_idx = idxs[:cut], idxs[cut:]
train_rows = [rows[i] for i in train_idx]
test_rows = [rows[i] for i in test_idx]
print(f'Train: {len(train_rows)} | Test: {len(test_rows)}')

letters = sorted(set(r['gold'] for r in train_rows))
letter_to_int = {l: i for i, l in enumerate(letters)}
print(f'Letters in train: {letters}')

d_stacks = {}
probe_info = {}
for L in PROBE_LAYERS:
    X_train = np.stack([r[f'act_L{L}'] for r in train_rows])
    y_train = np.array([letter_to_int[r['gold']] for r in train_rows])
    scaler = StandardScaler().fit(X_train)
    pca = PCA(n_components=min(64, X_train.shape[0] - 1), random_state=42).fit(scaler.transform(X_train))
    lr = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
        pca.transform(scaler.transform(X_train)), y_train)
    train_score = lr.score(pca.transform(scaler.transform(X_train)), y_train)
    X_test = np.stack([r[f'act_L{L}'] for r in test_rows])
    y_test = np.array([letter_to_int[r['gold']] if r['gold'] in letter_to_int else -1 for r in test_rows])
    mask = y_test >= 0
    if mask.sum() > 0:
        test_score = lr.score(pca.transform(scaler.transform(X_test[mask])), y_test[mask])
    else:
        test_score = 0.0
    probe_info[L] = dict(train=train_score, test=test_score)
    print(f'L{L}: train {train_score:.3f} | test {test_score:.3f}')

    # Build letter directions
    dirs = []
    for li in range(len(letters)):
        coef = lr.coef_[li]
        d_scaled = pca.components_.T @ coef
        d_raw = d_scaled * scaler.scale_
        d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
        dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
    d_stacks[L] = torch.stack(dirs)

print(f'\n✅ probes + directions for layers {PROBE_LAYERS}')

## Cell 6 — Install amnesic hooks per layer + run test eval

In [ ]:
class AmnesicHook:
    def __init__(self, layer_idx):
        self.layer_idx = layer_idx
        self.mode = 'off'
        self.prompt_len = None
    def set(self, prompt_len):
        self.mode = 'on'; self.prompt_len = prompt_len
    def off(self):
        self.mode = 'off'
    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off': return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            if T != self.prompt_len: return out
            hidden = hidden.clone()
            x = hidden[:, -1, :]
            ds = d_stacks[self.layer_idx]
            coefs = x @ ds.T
            delta = coefs @ ds
            hidden[:, -1, :] = x - delta  # α=1.0 full projection
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

hooks = {L: AmnesicHook(L) for L in PROBE_LAYERS}
handles = {L: get_layer_module(model, L).register_forward_hook(hooks[L].make_hook()) for L in PROBE_LAYERS}
print(f'amnesic hooks on {PROBE_LAYERS}')

# Eval on test set: baseline + amnesic at each layer
eval_results = []
for q in tqdm(test_rows, desc='eval'):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        prompt_len = ids.shape[1]
        row = dict(idx=q['idx'], gold=q['gold'], baseline_correct=q['baseline_correct'])
        row['baseline_pred'] = q['baseline_pred']
        # Amnesic at each probe layer
        for L in PROBE_LAYERS:
            for Lo in PROBE_LAYERS: hooks[Lo].off()
            hooks[L].set(prompt_len)
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {LL: logits[letter_ids[LL]].item() for LL in 'ABCDEFGHIJ'[:q['n_options']]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'amn_L{L}_pred'] = pred
            row[f'amn_L{L}_correct'] = (pred == q['gold'])
        for L in PROBE_LAYERS: hooks[L].off()
        eval_results.append(row)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

print(f'Eval done: {len(eval_results)} items')

## Cell 7 — Analysis + verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== Cross-arch amnesic on {MODEL_ID} ===')
print(f'N_test = {len(eval_results)}\n')

bc = [r['baseline_correct'] for r in eval_results]
base_acc = sum(bc) / len(eval_results) * 100
print(f'baseline    : {base_acc:.1f}%\n')
print(f'{"layer":>10s}  {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}  {"probe_train":>11s}  {"probe_test":>10s}')

stats = []
for L in PROBE_LAYERS:
    cc = [r[f'amn_L{L}_correct'] for r in eval_results]
    acc = sum(cc) / len(eval_results)
    delta = (acc - base_acc/100) * 100
    g = sum(1 for b, c in zip(bc, cc) if not b and c)
    l = sum(1 for b, c in zip(bc, cc) if b and not c)
    p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
    stats.append((L, acc*100, delta, g, l, p))
    star = ' ***' if delta>3 and l==0 else (' **' if delta>1.5 and l==0 else (' *' if delta>0.5 and l==0 else ''))
    print(f'{"L"+str(L):>10s}  {acc*100:5.1f}%  {delta:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}  {probe_info[L]["train"]:>11.3f}  {probe_info[L]["test"]:>10.3f}{star}')

best = max(stats, key=lambda s: s[2])
best_pareto = max([s for s in stats if s[4] == 0], key=lambda s: s[2], default=None)

print('\n=== Verdict ===')
print(f'Hybrid MoE reference (Qwen3.6-35B-A3B L17): +2.7pp Pareto (0 lost)\n')
if best_pareto and best_pareto[2] >= 2.7:
    print(f'✓ DENSE GENERALIZES: L{best_pareto[0]} gives {best_pareto[2]:+.1f}pp Pareto')
    print('  Amnesic is NOT hybrid-specific — paper has broad claim')
elif best_pareto and best_pareto[2] > 0.5:
    print(f'~ DENSE WEAKER: L{best_pareto[0]} gives {best_pareto[2]:+.1f}pp Pareto (vs +2.7pp hybrid)')
    print('  Effect reduced in dense — hybrid enables stronger amnesic')
elif best[2] > 0:
    print(f'~ DENSE NON-PARETO: best {best[2]:+.1f}pp with {best[4]} lost')
    print('  Effect present but noisier in dense')
else:
    print(f'✗ DENSE NULL/NEG: best {best[2]:+.1f}pp')
    print('  Amnesic is hybrid-specific — paper frames +2.7pp as arch-unique')

# Also report probe quality
print(f'\nProbe test accuracy per layer:')
for L in PROBE_LAYERS:
    print(f'  L{L}: {probe_info[L]["test"]:.3f} (train {probe_info[L]["train"]:.3f})')